In [1]:
import pandas as pd
import numpy as np

# Data Cleaning

### Global Variable

In [2]:
month_choices = [9, 10, 11, 12, 1, 2]

### Team Statistics

In [7]:
team_stats = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/team_stats.csv')

team_stats = team_stats.drop(columns='Unnamed: 0')
team_stats = team_stats.rename(columns={'Cmp-Att-Yd-TD-INT':'ManyStats'})
team_stats[['Completion', 'Attempts', 'Yards', 'Touchdowns', 'Interceptions']] = team_stats.ManyStats.str.split('-', expand=True)
team_stats = team_stats.drop(columns='ManyStats')

team_stats.columns = team_stats.columns.str.replace(' ', '_')
team_stats.columns = team_stats.columns.str.replace('.', '_')
team_stats.columns = team_stats.columns.str.replace('-', '_')

team_stats[['Success4thDown', 'Failed4thDown']] = team_stats.Fourth_Down_Conv_.str.split('-', expand=True)
team_stats[['Fumbles', 'FumblesLost']] = team_stats.Fumbles_Lost.str.split('-', expand=True)
team_stats[['Penalties', 'PenaltyYards']] = team_stats.Penalties_Yards.str.split('-', expand=True)
#team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats[['Sacked', 'SackYards']] = team_stats.Sacked_Yards.str.split('-', expand=True)
team_stats[['Success3rdDown', 'Failed3rdDown']] = team_stats.Third_Down_Conv_.str.split('-', expand=True)
team_stats[['TimeMinutes', 'TimeSeconds']] = team_stats.Time_of_Possession.str.split(':', expand=True)

team_stats = team_stats.drop(columns={'Fourth_Down_Conv_', 'Fumbles_Lost', 'Penalties_Yards', 'Sacked_Yards', 'Third_Down_Conv_', 'Time_of_Possession'})

team_stats['Rush_Yds_TDs'].str.count('-').value_counts()

team_stats.loc[2083]

team_stats.at[2083,'Rush_Yds_TDs'] = '8-neg1-0'

team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats = team_stats.drop(columns='Rush_Yds_TDs')

team_stats.at[2083, "RushYards"] = -1

skip_cols = ['team_name', 'date']
cols_to_change = [col for col in team_stats.columns if col not in skip_cols]
team_stats[cols_to_change] = team_stats[cols_to_change].astype(int)

team_stats['CompletionPct'] = round(100 * (team_stats['Completion'] / team_stats['Attempts']), 1)
team_stats['PctYardsPass'] = round(100 * (team_stats['Yards'] / team_stats['Total_Yards']), 1)
team_stats['PctYardsRush'] = round(100 * (team_stats['RushYards'] / team_stats['Total_Yards']), 1)
team_stats['SuccessRate4thDown'] = round(
    100 * (team_stats['Success4thDown'] / (team_stats['Success4thDown'] + team_stats['Failed4thDown'])), 1)
team_stats['SuccessRate3rdDown'] = round(
    100 * (team_stats['Success3rdDown'] / (team_stats['Success3rdDown'] + team_stats['Failed3rdDown'])), 1)
team_stats['PossessionTime'] = (60 * team_stats['TimeMinutes']) + team_stats['TimeSeconds']

team_stats[['DayofWeek', 'Month', 'DayofMonth', 'Year']] = team_stats.date.str.split(' ', expand=True)
team_stats['DayofMonth'] = team_stats['DayofMonth'].str.rstrip(',')

game_months = [
    (team_stats['Month'] == 'Sep'),
    (team_stats['Month'] == 'Oct'),
    (team_stats['Month'] == 'Nov'),
    (team_stats['Month'] == 'Dec'),
    (team_stats['Month'] == 'Jan'),
    (team_stats['Month'] == 'Feb')
]
month_choices = [9, 10, 11, 12, 1, 2]

team_stats['Month'] = np.select(game_months, month_choices, default=False)
team_stats['DayofMonth'] = team_stats['DayofMonth'].astype(int)
team_stats['Year'] = team_stats['Year'].astype(int)

date_mapping = {
    'Year':'year',
    'DayofMonth':'day',
    'Month':'month'}

team_stats = team_stats.rename(mapper=date_mapping, axis=1)
team_stats['gamedate'] = pd.to_datetime(team_stats[['year', 'month', 'day']])

team_stats = team_stats.drop(columns={'date','Completion','Success4thDown','Failed4thDown','Success3rdDown','Failed3rdDown', 'day'})

In [5]:
team_stats

,First_Downs,Net_Pass_Yards,Total_Yards,Turnovers,team_name,Attempts,Yards,Touchdowns,Interceptions,Fumbles,...,RushYards,RushTDs,CompletionPct,PctYardsPass,PctYardsRush,SuccessRate4thDown,SuccessRate3rdDown,PossessionTime,DayofWeek,gamedate
0,19,283,389,3,GNB,43,291,1,3,2,...,106,0,53.5,74.8,27.2,20.0,34.8,2082,Sunday,2022-11-06
1,19,137,254,1,DET,26,137,2,1,0,...,117,0,53.8,53.9,46.1,0.0,35.3,1518,Sunday,2022-11-06
2,20,127,338,2,PHI,28,154,1,1,2,...,211,2,53.6,45.6,62.4,50.0,15.8,2178,Sunday,2024-12-22
3,23,255,368,5,WAS,39,258,5,2,3,...,113,0,61.5,70.1,30.7,40.0,35.0,1422,Sunday,2024-12-22
4,23,204,311,0,DAL,41,204,2,0,0,...,107,1,65.9,65.6,34.4,NaN,30.4,1881,Sunday,2023-11-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,16,119,193,1,PIT,31,148,1,0,2,...,74,1,54.8,76.7,38.3,0.0,25.0,1320,Saturday,2025-01-04
2274,19,242,339,1,DAL,32,253,2,0,1,...,97,0,62.5,74.6,28.6,50.0,25.0,1862,Sunday,2023-12-24
2275,22,284,375,0,MIA,37,293,1,0,1,...,91,0,64.9,78.1,24.3,0.0,31.6,1738,Sunday,2023-12-24
2276,18,171,264,1,WAS,32,191,1,1,1,...,93,1,68.8,72.3,35.2,33.3,20.0,1616,Thursday,2024-11-14


### Offensive Player Statistics

In [5]:
player_offense = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/player_offense.csv')

new_offense_cols = ['randnumber','playername','team','completions','passattempts',
                    'passingyards','passingtds','interceptions','sackstaken','yardslostbysack',
                    'longestcompletion','passrating','rushingattempts','rushyards','rushtds',
                    'longestrush','targets','receptions','receivingyards','receivingtds',
                    'longestreception','fumbles','fumbleslost','gamedate']

player_offense.columns = new_offense_cols

realplayer = player_offense['playername'] != "Player"
playerhere = player_offense['playername'].notna()
player_offense = player_offense[realplayer]
player_offense = player_offense[playerhere]

player_offense[['dayofweek','month','day','year']] = player_offense.gamedate.str.split(' ', expand=True)
player_offense = player_offense.drop(columns='gamedate')
player_offense['day'] = player_offense['day'].str.rstrip(',')

game_months = [
    (player_offense['month'] == 'Sep'),
    (player_offense['month'] == 'Oct'),
    (player_offense['month'] == 'Nov'),
    (player_offense['month'] == 'Dec'),
    (player_offense['month'] == 'Jan'),
    (player_offense['month'] == 'Feb')
]

player_offense['month'] = np.select(game_months, month_choices, default=False)
player_offense['gamedate'] = pd.to_datetime(player_offense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_offense.columns if col not in skip_cols]
player_offense[cols_to_change] = player_offense[cols_to_change].astype(float)

player_offense['completionpct'] = round(100 * (player_offense['completions'] / player_offense['passattempts']), 1)
player_offense['td2int'] = round(player_offense['passingtds'] / player_offense['interceptions'], 3)
player_offense['yardsperrush'] = round(player_offense['rushyards'] / player_offense['rushingattempts'], 1)

player_offense = player_offense.drop(columns={'randnumber','day','month','year'})

/var/folders/xv/_79n8tfd6mx4j2z8m6rstygr0000gn/T/ipykernel_7114/3584193664.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  player_offense = player_offense[playerhere]


In [6]:
qbs = player_offense[player_offense['passattempts'] >= 10]
rbs = player_offense[(player_offense['rushingattempts'] >= 10) & (player_offense['passattempts'] <= 2)]
wrs = player_offense[player_offense['targets'] >= 2]

### Defensive Player Statistics

In [7]:
player_defense = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/player_defense.csv')

new_defense_cols = ['randnum','playername','team','interceptions','intrtnyards','int2td','intlongestyards',
                    'passdeflections','sacks','soloassisttackles','solotackles','assisttackles','tackelsforloss',
                    'qbhits','fumblerecovery','fumblereturnyards','fumble2td','forcedfumble','gamedate']
player_defense.columns = new_defense_cols

player_defense = player_defense[(player_defense['playername'] != 'Player') & player_defense['playername'].notna()]

player_defense[['dayofweek','month','day','year']] = player_defense.gamedate.str.split(' ', expand=True)
player_defense = player_defense.drop(columns='gamedate')
player_defense['day'] = player_defense['day'].str.rstrip(',')

game_months = [
    (player_defense['month'] == 'Sep'),
    (player_defense['month'] == 'Oct'),
    (player_defense['month'] == 'Nov'),
    (player_defense['month'] == 'Dec'),
    (player_defense['month'] == 'Jan'),
    (player_defense['month'] == 'Feb')
]

player_defense['month'] = np.select(game_months, month_choices, default=False)
player_defense['gamedate'] = pd.to_datetime(player_defense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_defense.columns if col not in skip_cols]
player_defense[cols_to_change] = player_defense[cols_to_change].astype(float)

player_defense = player_defense.drop(columns={'randnum','day','month','year'})

### Advanced Passing Statistics

In [8]:
advanced_passing = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/passing_advanced.csv')
advanced_passing = advanced_passing.drop(columns={'Unnamed: 0','Cmp','1D'})
new_advpass_cols = ['playername','team','attempts','passingyards','firstdownsperpassplay',
                    'intendairyards','intendairyardsperpassatt','completedairyards',
                    'completedairyardspercomp','completedairyardsperpassatt',
                    'yardsaftercatch','yacpercompletion','drops','percentdrops',
                    'badthrows','badthrowspercent','sackstaken','timesblitzed',
                    'timeshurried','hitstaken','timespressured','percentdbspressured',
                    'scrambles','yardsperscramble','date']

advanced_passing.columns = new_advpass_cols

advanced_passing = advanced_passing[advanced_passing['playername'] != 'Player']

advanced_passing['percentdbspressured'] = advanced_passing['percentdbspressured'].str.rstrip('%')
advanced_passing['badthrowspercent'] = advanced_passing['badthrowspercent'].str.rstrip('%')
advanced_passing['percentdrops'] = advanced_passing['percentdrops'].str.rstrip('%')

advanced_passing[['dayofweek','month','day','year']] = advanced_passing.date.str.split(' ', expand=True)
advanced_passing = advanced_passing.drop(columns='date')
advanced_passing['day'] = advanced_passing['day'].str.rstrip(',')

game_months = [
    (advanced_passing['month'] == 'Sep'),
    (advanced_passing['month'] == 'Oct'),
    (advanced_passing['month'] == 'Nov'),
    (advanced_passing['month'] == 'Dec'),
    (advanced_passing['month'] == 'Jan'),
    (advanced_passing['month'] == 'Feb')
]

advanced_passing['month'] = np.select(game_months, month_choices, default=False)
advanced_passing['gamedate'] = pd.to_datetime(advanced_passing[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_passing.columns if col not in skip_cols]
advanced_passing[cols_to_change] = advanced_passing[cols_to_change].astype(float)

advanced_passing = advanced_passing.drop(columns={'day','month','year'})

In [9]:
advanced_passing

,playername,team,attempts,passingyards,firstdownsperpassplay,intendairyards,intendairyardsperpassatt,completedairyards,completedairyardspercomp,completedairyardsperpassatt,...,sackstaken,timesblitzed,timeshurried,hitstaken,timespressured,percentdbspressured,scrambles,yardsperscramble,dayofweek,gamedate
0,Aaron Rodgers,GNB,43.0,291.0,25.0,429.0,10.0,182.0,7.9,4.2,...,1.0,10.0,1.0,4.0,6.0,12.8,3.0,13.7,Sunday,2022-11-06
2,Jared Goff,DET,26.0,137.0,38.5,106.0,4.1,44.0,3.1,1.7,...,0.0,15.0,1.0,3.0,4.0,15.4,0.0,NaN,Sunday,2022-11-06
3,Kenny Pickett,PHI,24.0,143.0,29.6,195.0,8.1,113.0,8.1,4.7,...,3.0,16.0,2.0,3.0,8.0,26.7,3.0,4.3,Sunday,2024-12-22
4,Jalen Hurts,PHI,4.0,11.0,25.0,40.0,10.0,5.0,5.0,1.3,...,0.0,1.0,0.0,0.0,0.0,0.0,2.0,14.0,Sunday,2024-12-22
6,Jayden Daniels,WAS,39.0,258.0,32.5,274.0,7.0,143.0,6.0,3.7,...,1.0,14.0,2.0,1.0,4.0,8.9,5.0,9.6,Sunday,2024-12-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3920,Russell Wilson,PIT,31.0,148.0,22.9,227.0,7.3,90.0,5.3,2.9,...,4.0,5.0,2.0,3.0,9.0,24.3,2.0,7.5,Saturday,2025-01-04
3921,Dak Prescott,DAL,32.0,253.0,25.0,177.0,5.5,141.0,7.1,4.4,...,4.0,9.0,3.0,7.0,14.0,36.8,2.0,12.0,Sunday,2023-12-24
3923,Tua Tagovailoa,MIA,37.0,293.0,42.1,362.0,9.8,197.0,8.2,5.3,...,1.0,6.0,1.0,3.0,5.0,13.2,0.0,NaN,Sunday,2023-12-24
3924,Jayden Daniels,WAS,32.0,191.0,28.6,114.0,3.6,45.0,2.0,1.4,...,3.0,1.0,0.0,0.0,3.0,7.7,4.0,3.8,Thursday,2024-11-14


### Advanced Receiving Statistics

In [10]:
advanced_receiving = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/receiving_advanced.csv')
advanced_receiving = advanced_receiving.drop(columns={'Unnamed: 0','Rec/Br'})
new_advrec_cols = ['playername','team','targets','receptions','yards','touchdowns',
                   'firstdownsreceiving','yardsbeforecatch','yardsbeforecatchper',
                   'yardsaftercatch','yardsaftercatchper','avgdeptoftarget',
                   'brokentackles','drops','droprate','intsontargets',
                   'passrtgontargets','date']

advanced_receiving.columns = new_advrec_cols

advanced_receiving = advanced_receiving[advanced_receiving['playername'] != 'Player']

advanced_receiving[['dayofweek','month','day','year']] = advanced_receiving.date.str.split(' ', expand=True)
advanced_receiving = advanced_receiving.drop(columns='date')
advanced_receiving['day'] = advanced_receiving['day'].str.rstrip(',')

game_months = [
    (advanced_receiving['month'] == 'Sep'),
    (advanced_receiving['month'] == 'Oct'),
    (advanced_receiving['month'] == 'Nov'),
    (advanced_receiving['month'] == 'Dec'),
    (advanced_receiving['month'] == 'Jan'),
    (advanced_receiving['month'] == 'Feb')
]

advanced_receiving['month'] = np.select(game_months, month_choices, default=False)
advanced_receiving['gamedate'] = pd.to_datetime(advanced_receiving[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_receiving.columns if col not in skip_cols]
advanced_receiving[cols_to_change] = advanced_receiving[cols_to_change].astype(float)

advanced_receiving = advanced_receiving.drop(columns={'day','month','year'})

### Advanced Rushing

In [11]:
advanced_rushing = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/rushing_advanced.csv')
advanced_rushing = advanced_rushing.drop(columns={'Unnamed: 0'})
new_advrush_cols = ['playername','team','rushes','attempts','touchdowns','firstdownsrushing',
                    'yardsbeforecontact','yardsbeforecontactper','yardsaftercontact',
                    'yardsaftercontactper','brokentackles','brokentacklesper','date']

advanced_rushing.columns = new_advrush_cols

advanced_rushing = advanced_rushing[advanced_rushing['playername'] != 'Player']

advanced_rushing[['dayofweek','month','day','year']] = advanced_rushing.date.str.split(' ', expand=True)
advanced_rushing = advanced_rushing.drop(columns='date')
advanced_rushing['day'] = advanced_rushing['day'].str.rstrip(',')

game_months = [
    (advanced_rushing['month'] == 'Sep'),
    (advanced_rushing['month'] == 'Oct'),
    (advanced_rushing['month'] == 'Nov'),
    (advanced_rushing['month'] == 'Dec'),
    (advanced_rushing['month'] == 'Jan'),
    (advanced_rushing['month'] == 'Feb')
]

advanced_rushing['month'] = np.select(game_months, month_choices, default=False)
advanced_rushing['gamedate'] = pd.to_datetime(advanced_rushing[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_rushing.columns if col not in skip_cols]
advanced_rushing[cols_to_change] = advanced_rushing[cols_to_change].astype(float)

advanced_rushing = advanced_rushing.drop(columns={'day','month','year'})

In [12]:
advanced_rushing

,playername,team,rushes,attempts,touchdowns,firstdownsrushing,yardsbeforecontact,yardsbeforecontactper,yardsaftercontact,yardsaftercontactper,brokentackles,brokentacklesper,dayofweek,gamedate
0,AJ Dillon,GNB,11.0,34.0,0.0,3.0,19.0,1.7,15.0,1.4,0.0,NaN,Sunday,2022-11-06
1,Aaron Jones,GNB,9.0,25.0,0.0,NaN,7.0,0.8,18.0,2.0,0.0,NaN,Sunday,2022-11-06
2,Aaron Rodgers,GNB,4.0,40.0,0.0,2.0,40.0,10.0,0.0,0.0,0.0,NaN,Sunday,2022-11-06
3,Kylin Hill,GNB,1.0,7.0,0.0,1.0,2.0,2.0,5.0,5.0,0.0,NaN,Sunday,2022-11-06
5,Jamaal Williams,DET,24.0,81.0,0.0,4.0,37.0,1.5,44.0,1.8,2.0,12.0,Sunday,2022-11-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10636,Jeremy McNichols,WAS,3.0,5.0,0.0,1.0,2.0,0.7,3.0,1.0,0.0,NaN,Thursday,2024-11-14
10637,Austin Ekeler,WAS,2.0,7.0,0.0,0.0,2.0,1.0,5.0,2.5,0.0,NaN,Thursday,2024-11-14
10639,Saquon Barkley,PHI,26.0,146.0,2.0,7.0,87.0,3.3,59.0,2.3,0.0,NaN,Thursday,2024-11-14
10640,Jalen Hurts,PHI,10.0,39.0,1.0,4.0,27.0,2.7,12.0,1.2,1.0,10.0,Thursday,2024-11-14


### Advanced Defensive Statistics

In [13]:
advanced_defense = pd.read_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse/defense_advanced.csv')
new_advdef_cols = ['randnum','playername','team','interceptions','deftargets','completions',
                   'cmppctincoverage','yardsallowed','yardsallowedpercmp',
                   'yardspertarget','tdsallowed','passrtgallowed',
                   'agvdepthoftargetcvg','airyardsoncmps','yardsaftercatch',
                   'blitzes','qbhurries','qbknockdown','sacks','pressures',
                   'soloassisttackles','missedtackles','missedtacklepct','date']
advanced_defense.columns = new_advdef_cols

advanced_defense = advanced_defense.drop(columns={'randnum', 'completions','airyardsoncmps'})

advanced_defense = advanced_defense[advanced_defense['playername'] != 'Player']

advanced_defense['cmppctincoverage'] = advanced_defense['cmppctincoverage'].str.rstrip('%')
advanced_defense['missedtacklepct'] = advanced_defense['missedtacklepct'].str.rstrip('%')

advanced_defense[['dayofweek','month','day','year']] = advanced_defense.date.str.split(' ', expand=True)
advanced_defense = advanced_defense.drop(columns='date')
advanced_defense['day'] = advanced_defense['day'].str.rstrip(',')

game_months = [
    (advanced_defense['month'] == 'Sep'),
    (advanced_defense['month'] == 'Oct'),
    (advanced_defense['month'] == 'Nov'),
    (advanced_defense['month'] == 'Dec'),
    (advanced_defense['month'] == 'Jan'),
    (advanced_defense['month'] == 'Feb')
]

advanced_defense['month'] = np.select(game_months, month_choices, default=False)
advanced_defense['gamedate'] = pd.to_datetime(advanced_defense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_defense.columns if col not in skip_cols]
advanced_defense[cols_to_change] = advanced_defense[cols_to_change].astype(float)

advanced_defense = advanced_defense.drop(columns={'day','month','year'})

In [14]:
advanced_defense

,playername,team,interceptions,deftargets,cmppctincoverage,yardsallowed,yardsallowedpercmp,yardspertarget,tdsallowed,passrtgallowed,...,blitzes,qbhurries,qbknockdown,sacks,pressures,soloassisttackles,missedtackles,missedtacklepct,dayofweek,gamedate
0,Keisean Nixon,GNB,0.0,5.0,60.0,42.0,14.0,8.4,0.0,87.1,...,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,Sunday,2022-11-06
1,Adrian Amos,GNB,0.0,3.0,66.7,8.0,4.0,2.7,0.0,70.1,...,0.0,0.0,0.0,0.0,0.0,5.0,1.0,16.7,Sunday,2022-11-06
2,Jaire Alexander,GNB,1.0,4.0,50.0,21.0,10.5,5.3,0.0,26.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,Sunday,2022-11-06
3,Krys Barnes,GNB,0.0,3.0,66.7,16.0,8.0,5.3,0.0,79.9,...,1.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,Sunday,2022-11-06
4,Darnell Savage Jr.,GNB,0.0,3.0,66.7,18.0,9.0,6.0,1.0,122.2,...,0.0,0.0,0.0,0.0,0.0,4.0,1.0,20.0,Sunday,2022-11-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33828,Nolan Smith,PHI,0.0,1.0,100.0,6.0,6.0,6.0,0.0,91.7,...,0.0,0.0,0.0,1.0,1.0,3.0,0.0,0.0,Thursday,2024-11-14
33829,C.J. Gardner-Johnson,PHI,0.0,2.0,50.0,5.0,5.0,2.5,1.0,95.8,...,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,Thursday,2024-11-14
33830,Brandon Graham,PHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,1.0,1.0,4.0,0.0,0.0,Thursday,2024-11-14
33831,Josh Sweat,PHI,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,1.0,1.0,3.0,0.0,0.0,Thursday,2024-11-14


#### Export to CSV
Ensuring that I don't have to rerun all these chunks of code each time

In [ ]:
team_stats.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/team_stats.csv')
qbs.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/qbs.csv')
rbs.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/rbs.csv')
wrs.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/wrs.csv')
player_defense.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/player_defense.csv')
advanced_passing.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/adv_pass.csv')
advanced_receiving.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/adv_pass.csv')
advanced_rushing.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/adv_rush.csv')
advanced_defense.to_csv('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/basicleaning/adv_def.csv')